# Librerías

In [18]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_iris
from torchviz import make_dot

# Iris dataset

In [ ]:
# En PyTorch, los datasets deben heredar de torch.utils.data.Dataset
class IrisDataset(Dataset):
    # Inicializamos el dataset cargando los datos y los ponemos en tensores
    def __init__(self):
        data = load_iris()
        self.X = torch.tensor(
            data.data, dtype=torch.float32
        )  # (150, 4)
        self.y = torch.tensor(
            data.target, dtype=torch.long
        )  # (150,)

    # PyTorch necesita saber el tamaño del dataset
    def __len__(self):
        return self.X.shape[0]
    
    # PyTorch necesita poder indexar el dataset
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
full_iris_dataset = IrisDataset()

In [ ]:
# ejemplos de muestras
for i in [1, 50, 100]:
    x, y = full_iris_dataset[i]
    print(f"Muestra {i}: x = {x}, y = {y}")

In [ ]:
len(full_iris_dataset)

# Definición de la red

In [ ]:
# Defino la red
class IrisRed(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 3)  # Solo hay una capa lineal, recibe vectores de dimesión 4 y devuelve vectores de dimensión 3 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(x)   # (B, 3)
        return h
    
# el softmax no se suele incluir en el modelo porque
# 1) la función de pérdida CrossEntropyLoss ya lo incluye
# 2) y porque en inferencia se suele usar argmax (selección de la clase con mayor puntuación)

# Grafo computacional

In [ ]:
# Instanciamos el modelo y el loss
# En PyTorch, el loss y el modelo suelen ser instanciados por separado
model = IrisRed()
criterion = nn.CrossEntropyLoss()

# Input dummy (batch size = 5)
x = torch.randn(5, 4, requires_grad=False)

# Forward
y = model(x)
# Loss
target = torch.tensor([1, 0, 2, 1, 2])  # clase verdadera
loss = criterion(y, target)


# Crear grafo computacional
dot = make_dot(loss, params=dict(model.named_parameters()))

# Mostrar o guardar
dot

# Stochastic Gradient Descent, dividr en batches

In [ ]:
# esta función devuelve un iterador sobre batches #DataLoader
dataloader = DataLoader(full_iris_dataset, batch_size=16, shuffle=True)
# por otro lado, la red se ha definido para procesar batches de datos

# Juntamos los tres ingredientes

In [ ]:
# instanciar modelo, loss y optimizador
# si se dispone de GPU, mover el modelo a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# la red se inicializa a pesos aleatorios cada vez que se instancia
model = IrisRed().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)

In [ ]:
# ALGORITMO DE ENTRANAMIENTO
num_epochs = 100
# durante un número de epochs
for epoch in range(1, num_epochs + 1):
    # poner el modelo en modo entrenamiento
    model.train()
    # para llevar la cuenta del loss
    running_loss = 0.0
    n_seen = 0
    for xB, yB in dataloader:
        xB, yB = xB.to(device), yB.to(device)
        optimizer.zero_grad()      # reset gradientes
        logits = model(xB)         # forward
        loss = criterion(logits, yB)
        loss.backward()            # backward
        optimizer.step()           # update
        running_loss += loss.item() * xB.size(0) # para llevar la cuenta del loss
        n_seen += xB.size(0)           # para llevar la cuenta del loss
    epoch_loss = running_loss / n_seen # loss medio en la epoch
    print(f"Epoch {epoch:03d} | train_loss = {epoch_loss:.4f}") # reportamos el loss

# Ejercicio

Divide el dataset en train y test, y entrena el modelo durante 100 epochs. Evalúa el accuracy en el conjunto de test.
- `torch.utils.data.random_split`
- `torch.argmax` con el parámetro `dim` (indica la dimensión a reducir). `dim` es equivalente a `axis` en NumPy. Seguntamente tengáis que usar `dim=1` para reducir la dimensión de las clases.

Recuerda poner el modelo en modo evaluación con `model.eval()` cuando evalúes en el conjunto de test.

In [15]:
# Ejercicio
# Conjunto de entrenar train --> codigo anterior
# Conjunto de evaluar test hits/test
# torch.utils.data.random_split Divido el dataset 70% train 30% test

# =========================
# 1. Split dataset
# =========================

from torch.utils.data import random_split

train_size = int(0.7 * len(full_iris_dataset))
test_size  = int(0.3 * len(full_iris_dataset))

train_iris_dataset, test_iris_dataset = random_split(
    full_iris_dataset,
    [train_size, test_size]
)

train_loader = DataLoader(train_iris_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_iris_dataset, batch_size=16, shuffle=True)

# =========================
# 2. Modelo, loss y optimizador
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = IrisRed().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)


In [16]:
# =========================
# 3. Training loop
# =========================

num_epochs = 100
# durante un número de epochs
for epoch in range(1, num_epochs + 1):
    # poner el modelo en modo entrenamiento
    model.train()
    # para llevar la cuenta del loss
    running_loss = 0.0
    n_seen = 0
    for xB, yB in train_loader:
        xB, yB = xB.to(device), yB.to(device)
        optimizer.zero_grad()      # reset gradientes
        logits = model(xB)         # forward
        loss = criterion(logits, yB)
        loss.backward()            # backward
        optimizer.step()           # update
        running_loss += loss.item() * xB.size(0) # para llevar la cuenta del loss
        n_seen += xB.size(0)           # para llevar la cuenta del loss
    epoch_loss = running_loss / n_seen # loss medio en la epoch
    print(f"Epoch {epoch:03d} | train_loss = {epoch_loss:.4f}") # reportamos el loss



Epoch 001 | train_loss = 1.9394
Epoch 002 | train_loss = 1.3178
Epoch 003 | train_loss = 1.2265
Epoch 004 | train_loss = 1.1608
Epoch 005 | train_loss = 1.1062
Epoch 006 | train_loss = 1.0599
Epoch 007 | train_loss = 1.0109
Epoch 008 | train_loss = 0.9681
Epoch 009 | train_loss = 0.9342
Epoch 010 | train_loss = 0.8947
Epoch 011 | train_loss = 0.8664
Epoch 012 | train_loss = 0.8367
Epoch 013 | train_loss = 0.8122
Epoch 014 | train_loss = 0.7911
Epoch 015 | train_loss = 0.7640
Epoch 016 | train_loss = 0.7513
Epoch 017 | train_loss = 0.7274
Epoch 018 | train_loss = 0.7156
Epoch 019 | train_loss = 0.6965
Epoch 020 | train_loss = 0.6844
Epoch 021 | train_loss = 0.6673
Epoch 022 | train_loss = 0.6632
Epoch 023 | train_loss = 0.6466
Epoch 024 | train_loss = 0.6354
Epoch 025 | train_loss = 0.6249
Epoch 026 | train_loss = 0.6165
Epoch 027 | train_loss = 0.6101
Epoch 028 | train_loss = 0.6053
Epoch 029 | train_loss = 0.5949
Epoch 030 | train_loss = 0.5845
Epoch 031 | train_loss = 0.5807
Epoch 03

In [17]:
# =========================
# 4. Evaluación
# =========================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        outputs = model(x_batch)

        # ⭐ IMPORTANTE: dim=1
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total

print(f"Accuracy test: {accuracy:.4f}")

Accuracy test: 0.9333
